In [1]:
# SparkSession para JDBC
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import datetime

spark = SparkSession.builder \
    .master("spark://spark:7077") \
    .appName("DeltaWrite") \
    .config("spark.jars",
            "/opt/spark/jars-extra/delta-spark_2.12-3.0.0.jar,"
            "/opt/spark/jars-extra/delta-storage-3.0.0.jar,"
            "/opt/spark/jars-extra/hadoop-aws-3.3.4.jar,"
            "/opt/spark/jars-extra/aws-java-sdk-bundle-1.12.367.jar,"
            "/opt/spark/jars-extra/mssql-jdbc-12.6.0.jre11.jar") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "sparkuser") \
    .config("spark.hadoop.fs.s3a.secret.key", "sparkpass123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.fast.upload", "true") \
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "bytebuffer") \
    .config("spark.hadoop.fs.s3a.multipart.size", "64M") \
    .config("spark.hadoop.fs.s3a.threads.max", "64") \
    .getOrCreate()

print("--------------------------")
print("Spark version:", spark.version)
print("Hora:", datetime.datetime.now())
print("--------------------------")

26/06/14 06:04:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


--------------------------
Spark version: 3.5.0
Hora: 2026-06-14 06:04:39.416169
--------------------------


In [2]:
# Leer Tabla del SQL Server BLL usando el Secret desde Hashicorp Vault
import hvac

client = hvac.Client(
    url="http://vault.luispicado.com",
    token="REDACTED_VAULT_TOKEN"
)

# Leer el secreto desde el engine "secret"
secret = client.secrets.kv.v2.read_secret_version(
    path="mssql",
    mount_point="secret",
    raise_on_deleted_version=True
)

user = secret["data"]["data"]["username"]
pwd = secret["data"]["data"]["password"]

print("--------------------------")
print("Credenciales")
print("--------------------------")
print("Usuario:", user)
print("Password:", pwd)
print("--------------------------")
print("Hora:", datetime.datetime.now())
print("--------------------------")

--------------------------
Credenciales
--------------------------
Usuario: sql_userbll
Password: bll.1234!
--------------------------
Hora: 2026-06-14 06:05:36.447042
--------------------------


In [3]:
# Cadena de conexion
jdbc_url = (
    f"jdbc:sqlserver://msql.luispicado.com:51430;"
    f"databaseName=AdventureWorks2022;"
    f"user={user};password={pwd};"
    f"encrypt=true;trustServerCertificate=true"
)

df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Sales.SalesOrderHeader") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

print("--------------------------")
print("Filas leídas desde SQL Server:", df.count())
print("Hora:", datetime.datetime.now())
print("--------------------------")

--------------------------


[Stage 0:>                                                          (0 + 1) / 1]

Filas leídas desde SQL Server: 31465
Hora: 2026-06-14 06:05:43.960421
--------------------------


In [4]:
# Guardar como Delta
delta_path = "s3a://bronce/delta/sales/SalesOrderHeader"

# Para probar con solo 1 archivo
df.repartition(1) \
  .write \
  .format("delta") \
  .mode("overwrite") \
  .save(delta_path)

print("--------------------------")
print("Escritura Delta:", delta_path)
print("Hora:", datetime.datetime.now())
print("--------------------------")

26/06/14 06:05:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/14 06:05:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
[Stage 9:===================================>                    (32 + 18) / 50]

--------------------------
Escritura Delta: s3a://bronce/delta/sales/SalesOrderHeader
Hora: 2026-06-14 06:05:59.979022
--------------------------


In [7]:
# Leer como Delta
delta_path = "s3a://bronce/delta/sales/SalesOrderHeader"

df_delta = spark.read \
    .format("delta") \
    .load(delta_path)

print("Schema:")
df_delta.printSchema()

print("Primeras 10 filas:")
df_delta.show(10, truncate=False)

Schema:
root
 |-- SalesOrderID: integer (nullable = true)
 |-- RevisionNumber: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- ShipDate: timestamp (nullable = true)
 |-- Status: integer (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- SalesOrderNumber: string (nullable = true)
 |-- PurchaseOrderNumber: string (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- BillToAddressID: integer (nullable = true)
 |-- ShipToAddressID: integer (nullable = true)
 |-- ShipMethodID: integer (nullable = true)
 |-- CreditCardID: integer (nullable = true)
 |-- CreditCardApprovalCode: string (nullable = true)
 |-- CurrencyRateID: integer (nullable = true)
 |-- SubTotal: decimal(19,4) (nullable = true)
 |-- TaxAmt: decimal(19,4) (nullable = true)
 |-- Freight: decimal(1